# Todo: Phase 1 Notebook

*Go in the folder ```\matching_project``` and type ```pip install -e .``` so you won't have issues to import librairies that are not in the same folder as our code (e.g. to import ```from core.instance import Instance``` in ```gale_shapley.py```).*

### Import of the librairies 

---

In [4]:
from data import create_synthetic_instance
from mechanisms.serial_dictatorship import serial_dictatorship
from mechanisms.gale_shapley import gale_shapley
from mechanisms.ttc_from_matching import ttc_from_matching
from metrics.overall_insatisfaction import overall_insatisfaction

### Simple Example 

---

In [5]:
# Generating simple test data 
n_students = 4
n_projects = 4
capacity = 1
phi = 0.5
test_instance = create_synthetic_instance(n_students=n_students, n_projects=n_projects, capacity=capacity, phi=phi)

In [6]:
sd_test = serial_dictatorship(test_instance)

print(sd_test)

=== Student -> Project ===
Student 0 -> Project 0
Student 1 -> Project 1
Student 2 -> Project 2
Student 3 -> Project 3

=== Project -> Students ===
Project 0 -> Students [0]
Project 1 -> Students [1]
Project 2 -> Students [2]
Project 3 -> Students [3]


In [7]:
gs_test = gale_shapley(test_instance)

print(gs_test)

=== Student -> Project ===
Student 0 -> Project 0
Student 1 -> Project 1
Student 2 -> Project 2
Student 3 -> Project 3

=== Project -> Students ===
Project 0 -> Students [0]
Project 1 -> Students [1]
Project 2 -> Students [2]
Project 3 -> Students [3]


In [8]:
ttc_test = ttc_from_matching(sd_test)

print(ttc_test)

=== Student -> Project ===
Student 0 -> Project 0
Student 1 -> Project 1
Student 2 -> Project 2
Student 3 -> Project 3

=== Project -> Students ===
Project 0 -> Students [0]
Project 1 -> Students [1]
Project 2 -> Students [2]
Project 3 -> Students [3]


### Pathological Example 

--- 
Shows the differences between : 
- Serial dictatorship
- Gale-Shapley algorithm 
- Top Trading Cycles after Gale-Shapley

In [9]:
# Librairies in order to create the pathological example 
from collections import deque 
from core.instance import Instance, Student, Project
from core.matching import Matching

In [10]:
A, B, C, D = 100, 101, 102, 103

students = (
    Student(id=0),
    Student(id=1),
    Student(id=2),
    Student(id=3)
)
 
projects = (
    Project(id=A, capacity=1),
    Project(id=B, capacity=1),
    Project(id=C, capacity=1),
    Project(id=D, capacity=1)
)
 
preferences = {
    0: [A, C, D, B],   # 0 :  A > B > C > D
    1: [A, B, C, D],   
    2: [C, D, A, B],   
    3: [B, D, A, C]
}

# Heart of the pathological example : each project has its own priorities contrary to the GPA version
school_priorities = {
    A: [0, 1, 2, 3],
    B: [1, 0, 2, 3],
    C: [2, 3, 0, 1], 
    D: [3, 2, 0, 1]
}

instance = Instance(students = students, projects=projects, preferences=preferences, school_priorities = school_priorities)

In [11]:
sd_pathological = serial_dictatorship(instance, order=[1, 0, 3, 2])
print(sd_pathological)

=== Student -> Project ===
Student 1 -> Project 100
Student 0 -> Project 102
Student 3 -> Project 101
Student 2 -> Project 103

=== Project -> Students ===
Project 100 -> Students [1]
Project 101 -> Students [3]
Project 102 -> Students [0]
Project 103 -> Students [2]


In [12]:
gs_pathological = gale_shapley(instance)
print(gs_pathological)

=== Student -> Project ===
Student 0 -> Project 100
Student 1 -> Project 101
Student 2 -> Project 102
Student 3 -> Project 103

=== Project -> Students ===
Project 100 -> Students [0]
Project 101 -> Students [1]
Project 102 -> Students [2]
Project 103 -> Students [3]


In [13]:
# Comparison : 
sd_overall_insatisfaction = overall_insatisfaction(sd_pathological, school_priorities=True)
gs_overall_insatisfaction = overall_insatisfaction(gs_pathological, school_priorities=True)

print(" --- Serial Dictatorship --- ")
print(f" Student insatisfaction : {sd_overall_insatisfaction["students"]}")
print(f" Projects insatisfaction : {sd_overall_insatisfaction["projects"]}")
print(f" Total insatisfaction : {sd_overall_insatisfaction["students"] + sd_overall_insatisfaction["projects"]}")

print()

print(" --- Gale Shapley Alogorithm --- ")
print(f" Student insatisfaction : {gs_overall_insatisfaction["students"]}")
print(f" Projects insatisfaction : {gs_overall_insatisfaction["projects"]}")
print(f" Total insatisfaction : {gs_overall_insatisfaction["students"] + gs_overall_insatisfaction["projects"]}")

 --- Serial Dictatorship --- 
 Student insatisfaction : 0.5
 Projects insatisfaction : 1.75
 Total insatisfaction : 2.25

 --- Gale Shapley Alogorithm --- 
 Student insatisfaction : 0.5
 Projects insatisfaction : 0.0
 Total insatisfaction : 0.5


Example for TTC 

---

In [14]:
students = (Student(id=1), Student(id=2), Student(id=3))
projects = (Project(id=10, capacity=1), Project(id=11, capacity=1), Project(id=12, capacity=1))
preferences = {
    1: [10, 11, 12],
    2: [10, 12, 11],
    3: [12, 10, 11],
}
instance = Instance(students=students, projects=projects, preferences=preferences)
unoptimized_matching = Matching(instance)

unoptimized_matching.assign(1, 11)
unoptimized_matching.assign(2, 12)
unoptimized_matching.assign(3, 10)

In [15]:
unopti_overall_insatisfaction = overall_insatisfaction(unoptimized_matching)
ttc_overall_insatisfaction = overall_insatisfaction(ttc_from_matching(unoptimized_matching))


print(" --- Unoptimised matching --- ")
print(f" Student insatisfaction : {sd_overall_insatisfaction["students"]}")
print(f" Projects insatisfaction : {sd_overall_insatisfaction["projects"]}")
print(f" Total insatisfaction : {sd_overall_insatisfaction["students"] + sd_overall_insatisfaction["projects"]}")

print()

print(" --- Top Trading Cycles --- ")
print(f" Student insatisfaction : {gs_overall_insatisfaction["students"]}")
print(f" Projects insatisfaction : {gs_overall_insatisfaction["projects"]}")
print(f" Total insatisfaction : {gs_overall_insatisfaction["students"] + gs_overall_insatisfaction["projects"]}")

 --- Unoptimised matching --- 
 Student insatisfaction : 0.5
 Projects insatisfaction : 1.75
 Total insatisfaction : 2.25

 --- Top Trading Cycles --- 
 Student insatisfaction : 0.5
 Projects insatisfaction : 0.0
 Total insatisfaction : 0.5
